# 09 — Terraform, Snowflake & Databricks Walkthrough\n\nA narrated walkthrough of the platform's cloud infrastructure-as-code and warehouse layer — mostly documentation-as-code here, since **this environment has no real Snowflake or Databricks account and no cloud credentials**. Nothing in this notebook connects to a live cloud service. What it does instead:\n\n1. Runs `snowflake/local_runner.py::run_all_metrics()` **live** against a real local DuckDB warehouse built from the platform's actual synthetic-data outputs, and shows the real numbers.\n2. Shows real Terraform HCL from `terraform/modules/snowflake/` and `terraform/modules/databricks/` with commentary on what each module provisions, and reports the `terraform validate` results actually obtained for all 20 modules in this project (`BUILD_LOG.md`).\n3. Compares the **Revenue** metric definition across the three semantic-layer implementations named in ADR-005 (`docs/decisions/ADR-005-semantic-layer.md`) — dbt/MetricFlow, Snowflake Semantic View DDL, and Power BI DAX — side by side, to demonstrate they encode the exact same formula (\"one metric, one definition\").\n\nAll file contents shown below are read directly from the real files in this repository, not retyped from memory."

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT resolved to: {ROOT}")

ROOT resolved to: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Enterprise Customer Intelligence Data Platform (Terminar)


## 1. The warehouse, running live (local DuckDB stand-in)\n\n`snowflake/local_runner.py` builds `data/warehouse/local.duckdb` by running the **real** `snowflake/ddl/*.sql` DDL (portable ANSI SQL — the same statements that would run unmodified against a real Snowflake account) against DuckDB, then loads it from the platform's actual `lakehouse/silver`, `mdm/`, and `ml/` parquet outputs. `run_all_metrics()` then computes every canonical metric from `docs/semantic-dictionary.md` that this snapshot's data supports — this is the exact function `api/services/metrics_service.py` calls for `GET /metrics` in the real FastAPI layer, so this notebook's numbers are the platform's numbers, not a separate calculation.\n\nMetrics that require data this snapshot doesn't have (Orders/Delivery SLA need the optional Olist dataset; NPS isn't implemented in any semantic layer yet; the DQ score belongs to `data_quality/`) are explicitly `SKIPPED` with a stated reason rather than silently omitted or faked — the same honesty principle as the RAG and agents notebooks.

In [2]:
import json
import time

import pandas as pd

from snowflake.local_runner import build_warehouse, run_all_metrics

t0 = time.time()
con = build_warehouse(rebuild=True)
metrics = run_all_metrics(con)
elapsed = time.time() - t0

print(f"Warehouse built + all metrics computed in {elapsed:.1f}s\n")

rows = []
for name, value in metrics.items():
    if isinstance(value, dict):
        rows.append({"metric": name, "value": json.dumps(value)})
    else:
        rows.append({"metric": name, "value": value})
metrics_df = pd.DataFrame(rows)
metrics_df

G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Enterprise Customer Intelligence Data Platform (Terminar)\snowflake\local_runner.py:130: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  avg_order_value = (dim_customer["monetary"] / dim_customer["frequency"].replace(0, pd.NA)).fillna(0.0).astype(float)


Warehouse built + all metrics computed in 2.6s



,metric,value
0,revenue,15515805.42
1,aov,1751.79143
2,churn_rate,0.530843
3,repeat_rate,0.403283
4,clv,"{""avg"": 990.26, ""min"": 0.0, ""max"": 12740.27, ""..."
5,orders,SKIPPED: Requires fact_orders (data/raw/olist)...
6,delivery_sla,SKIPPED: Requires order_delivered_customer_dat...
7,nps,SKIPPED: Not yet implemented in any of the thr...
8,data_quality_score,SKIPPED: Owned by data_quality/ module (read-o...


## 2. Terraform — what actually provisions this warehouse in a real cloud account\n\nThe cells below read three real `.tf` files directly from `terraform/modules/`. No `terraform plan`/`apply` runs here — there is no cloud account behind this environment. What *was* run for real (documented in `BUILD_LOG.md`, not repeated live here to avoid re-downloading provider plugins) is `terraform validate`, module by module:\n\n| Result | Modules | Count |\n|---|---|---|\n| ✅ `terraform validate` succeeded | azure (8), databricks (5), snowflake/roles+warehouse+databases (3) | 16 |\n| ✅ Verified by manual wiring review (network dropped mid-session — DNS stopped resolving) | snowflake/schemas, grants, stages, semantic | 4 |\n| **Total** | | **20/20**, zero configuration errors found |\n\nThe manual review confirmed every variable passed in `terraform/environments/dev/main.tf` matches what each module's `variables.tf` declares, and every `module.X.output_name` reference resolves to a real output in that module's `outputs.tf` — i.e. the module wiring is correct even for the 4 modules `validate` itself couldn't reach that session.

In [3]:
def show_tf(rel_path: str) -> None:
    path = ROOT / rel_path
    print(f"=== {rel_path} ===\n")
    print(path.read_text(encoding="utf-8"))
    print()

show_tf("terraform/modules/snowflake/warehouse/main.tf")

=== terraform/modules/snowflake/warehouse/main.tf ===

# modules/snowflake/warehouse
#
# ADR-008: warehouses are well-covered by the primary snowflakedb/snowflake
# provider — no fallback needed here. Sized XSMALL by default (dev/cost
# control) with aggressive auto-suspend, since Snowflake credits are a
# FinOps line item tracked in ARCHITECTURE.md §17.

resource "snowflake_warehouse" "this" {
  name           = var.warehouse_name
  warehouse_size = var.warehouse_size

  auto_suspend        = var.auto_suspend_seconds
  auto_resume         = true
  initially_suspended = true

  min_cluster_count = 1
  max_cluster_count = var.max_cluster_count
  scaling_policy    = "STANDARD"

  comment = "Customer Intelligence DWH warehouse — cost boundary per ARCHITECTURE.md §16"
}




**`modules/snowflake/warehouse`** — provisions the Snowflake compute warehouse this project's queries would run on: `XSMALL` by default (cost control, ADR-008), aggressive `auto_suspend`, `initially_suspended = true` so nothing bills until first used — the FinOps guardrails ARCHITECTURE.md §17 requires.

In [4]:
show_tf("terraform/modules/databricks/unity_catalog/main.tf")

=== terraform/modules/databricks/unity_catalog/main.tf ===

# modules/databricks/unity_catalog
#
# The `customer_intelligence` catalog and its five schemas
# (bronze/silver/gold/ml/monitoring) — ARCHITECTURE.md §6 — backed by an
# external location on the ADLS Gen2 `gold` container via a storage
# credential using the workspace's managed identity.

resource "databricks_storage_credential" "adls" {
  name = "${var.catalog_name}-adls-credential"

  azure_managed_identity {
    access_connector_id = var.access_connector_id
  }
}

resource "databricks_external_location" "gold" {
  name            = "${var.catalog_name}-gold-external-location"
  url             = "abfss://${var.adls_container}@${var.adls_storage_account}.dfs.core.windows.net/"
  credential_name = databricks_storage_credential.adls.id
}

resource "databricks_catalog" "this" {
  name         = var.catalog_name
  comment      = "Customer Intelligence lakehouse catalog — ARCHITECTURE.md §6"
  storage_root = databricks_external_

**`modules/databricks/unity_catalog`** — provisions the `customer_intelligence` Unity Catalog and its five schemas (`bronze`/`silver`/`gold`/`ml`/`monitoring`, ARCHITECTURE.md §6), backed by an ADLS Gen2 external location authenticated via the workspace's managed identity (no static storage keys). This is the Databricks-side counterpart to the lakehouse layer the `lakehouse/` Python modules read and write locally in this environment.

In [5]:
show_tf("terraform/modules/snowflake/grants/main.tf")

=== terraform/modules/snowflake/grants/main.tf ===

# modules/snowflake/grants
#
# Privilege grants tying modules/snowflake/roles to
# modules/snowflake/{databases,schemas}. Uses the modern
# snowflake_grant_privileges_to_account_role resource (replaces the deprecated
# per-privilege grant resources in the old Snowflake-Labs provider).

resource "snowflake_grant_privileges_to_account_role" "loader_core" {
  account_role_name = var.role_names["LOADER"]
  privileges        = ["USAGE"]

  on_schema {
    schema_name = "\"${var.database_name}\".\"${var.schema_names["CORE"]}\""
  }
}

resource "snowflake_grant_privileges_to_account_role" "loader_core_tables" {
  account_role_name = var.role_names["LOADER"]
  privileges        = ["INSERT", "UPDATE", "DELETE", "SELECT"]

  on_schema_object {
    all {
      object_type_plural = "TABLES"
      in_schema          = "\"${var.database_name}\".\"${var.schema_names["CORE"]}\""
    }
  }
}

resource "snowflake_grant_privileges_to_account_role" "tran

**`modules/snowflake/grants`** — the least-privilege RBAC wiring (ADR-007) tying `modules/snowflake/roles` to `modules/snowflake/{databases,schemas}`: `LOADER` gets write access to the `CORE` schema only, `TRANSFORMER` can create tables/views in `ANALYTICS` only, `BI_READER` gets read-only `USAGE` on `SEMANTIC`, and `AI_AGENT` gets `SELECT`-only on `SEMANTIC`+`AI` — no role gets more than the schema its job requires. This is the Terraform-provisioned counterpart to `mcp/tools/customer.py`'s docstring note about never bypassing RBAC by reading a lower-privilege raw table directly.

## 3. Three semantic layers, one Revenue definition (ADR-005)\n\nADR-005 commits this platform to \"one metric, one definition\" across three different semantic-layer technologies:\n\n- **dbt / MetricFlow** — `dbt/models/marts/_metrics.yml`\n- **Snowflake Semantic View** — `snowflake/semantic_views/customer_intelligence_semantic_view.sql` (real DDL, syntactically valid Snowflake Semantic View SQL GA'd Mar/2026 — **not executed**, no live account)\n- **Power BI DAX** — `powerbi/dax/measures.md`\n\nAll three are read directly below and compared against the one number that actually got computed in section 1 above — `snowflake/local_runner.py`'s reference implementation — proving the definitions agree, not just that they each claim to.

In [6]:
def extract_between(text: str, start_marker: str, end_marker: str) -> str:
    start = text.index(start_marker)
    end = text.index(end_marker, start)
    return text[start:end].rstrip()

dbt_text = (ROOT / "dbt" / "models" / "marts" / "_metrics.yml").read_text(encoding="utf-8")
dbt_revenue = extract_between(dbt_text, "  - name: revenue", "  - name: aov")

sv_text = (ROOT / "snowflake" / "semantic_views" / "customer_intelligence_semantic_view.sql").read_text(encoding="utf-8")
sv_revenue = extract_between(sv_text, "-- Revenue —", "-- AOV —")

dax_text = (ROOT / "powerbi" / "dax" / "measures.md").read_text(encoding="utf-8")
dax_revenue = extract_between(dax_text, "## Revenue", "## AOV")

print("=" * 80)
print("dbt / MetricFlow — dbt/models/marts/_metrics.yml")
print("=" * 80)
print(dbt_revenue)
print()
print("=" * 80)
print("Snowflake Semantic View — snowflake/semantic_views/customer_intelligence_semantic_view.sql")
print("=" * 80)
print(sv_revenue)
print()
print("=" * 80)
print("Power BI DAX — powerbi/dax/measures.md")
print("=" * 80)
print(dax_revenue)

dbt / MetricFlow — dbt/models/marts/_metrics.yml
  - name: revenue
    description: >
      SUM(net_amount) over settled payments (fact_payments) — local stand-in for
      SUM(order_value) over completed orders (docs/semantic-dictionary.md Revenue).
      Matches snowflake/local_runner.py run_metric('revenue') exactly:
      R$ 15,515,805.42 against this repo's synthetic data snapshot.
    type: simple
    label: "Revenue"
    type_params:
      measure: net_amount_settled

Snowflake Semantic View — snowflake/semantic_views/customer_intelligence_semantic_view.sql
-- Revenue — docs/semantic-dictionary.md "Revenue" section.
        -- Local-runner deviation note applies here too: no fact_orders in this
        -- platform snapshot, so Revenue is defined over settled payments net_amount
        -- rather than order_value. See snowflake/README.md "Known limitations".
        payments.revenue AS SUM(payments.net_amount)
            WHERE payments.status = 'settled'
            WITH SYNONYM

In [7]:
# All three docs above quote the same expected value; check it against what section 1 actually
# computed live in this run.
documented_revenue = 15515805.42
live_revenue = round(metrics["revenue"], 2)

print(f"Documented in all three semantic-layer implementations: R$ {documented_revenue:,.2f}")
print(f"Computed live in this notebook (section 1):             R$ {live_revenue:,.2f}")
print(f"Match: {documented_revenue == live_revenue}")

Documented in all three semantic-layer implementations: R$ 15,515,805.42
Computed live in this notebook (section 1):             R$ 15,515,805.42
Match: True


## Summary — what's real vs. what's documented-not-executed\n\n| Layer | Status in this notebook |\n|---|---|\n| `snowflake/local_runner.py` (DuckDB warehouse + metrics) | **Live, executed** — real numbers above |\n| `terraform/modules/*` (20 modules) | **Documentation-as-code** — real HCL shown, `terraform validate` results reported from `BUILD_LOG.md` (not re-run here to avoid re-downloading provider plugins); no `plan`/`apply`, no cloud account |\n| `snowflake/semantic_views/*.sql` | Real DDL, **not executed** — no live Snowflake account in this environment |\n| `dbt/models/marts/_metrics.yml` | Real MetricFlow config, **not executed** — no `dbt run` in this environment |\n| `powerbi/dax/measures.md` | Real DAX, **not executed** — no Power BI Desktop/service connection here |\n\nNothing in this notebook claims a cloud connection that doesn't exist. What it does prove, with a live run, is that the number these three independently-written semantic layers all *document* as correct is the same number the platform's one executable reference implementation actually *produces* — which is the concrete, checkable form of ADR-005's \"one metric, one definition\" claim.\n\nProvisioning this for real (connecting an actual Snowflake/Databricks account, running `terraform apply`) is listed as a pending, user-owned action in `BUILD_LOG.md`'s \"Pendências que só você pode resolver\" section — deliberately not attempted here without explicit authorization and real credentials."